In [66]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random, math, copy
import importlib
import nn
importlib.reload(nn)
from nn import NeuralNetwork

In [68]:
multimodal_train = pd.read_csv("dane_ae3/multimodal-large-training.csv")
multimodal_train.head()
X_multimodal_train = multimodal_train['x'].values.reshape(-1, 1)
y_multimodal_train = multimodal_train['y'].values.reshape(-1, 1)

multimodal_test = pd.read_csv("dane_ae3/multimodal-large-test.csv")
multimodal_test.head()
X_multimodal_test = multimodal_test['x'].values.reshape(-1, 1)
y_multimodal_test = multimodal_test['y'].values.reshape(-1, 1)

In [69]:
nn_test = NeuralNetwork(X_multimodal_train, y_multimodal_train, [1, 32, 32, 32, 1],
                              activation_fun="sigmoid", output_activation="linear", loss_fun="mse")

In [70]:
weights, biases = nn_test.get_weights()

In [93]:
class NeuralNetworkEvolution:
    def __init__(self, architecture, X, Y,
                activation_fun, output_activation, loss_fun,
                population_size=25):
        self.architecture = architecture
        self.population_size = population_size
        self.activation_fun = activation_fun
        self.output_activation = output_activation
        self.loss_fun = loss_fun
        self.X = X
        self.Y = Y
        self.generation = self.generate_population(population_size)
        self.best_individual = None
        self.best_fitness = float('inf')
        self.fitness_history = []

    def generate_population(self, population_size):
        population = []
        for _ in range(population_size):
            nn = self.generate_individual()
            population.append(nn)
        return population
    
    def generate_individual(self):
        nn = NeuralNetwork(self.X, self.Y, self.architecture,
                            activation_fun=self.activation_fun,
                            output_activation=self.output_activation,
                            loss_fun=self.loss_fun)
        return nn
    
    def _evaluate_fitness(self, individual):
        predictions = individual.predict(self.X)
        loss = individual.loss(predictions, self.Y)
        return loss
    
    def _crossover(self, parent1, parent2):
        weights1, biases1 = parent1.get_weights()
        weights2, biases2 = parent2.get_weights()

        new_weights = []
        new_biases = []

        for i in range(len(weights1)):
            w1, w2 = weights1[i], weights2[i]
            b1, b2 = biases1[i], biases2[i]

            if i == len(weights1) - 1:
                # Ostatnia warstwa: crossover po wierszach
                layer_size = w1.shape[0]
                crossover_point = random.randint(1, layer_size - 1)
                new_w = np.vstack([
                    w1[:crossover_point, :],
                    w2[crossover_point:, :]
                ])
                new_b = np.concatenate([
                    b1[:, :crossover_point],
                    b2[:, crossover_point:]
                ], axis=1)
            else:
                # Wcześniejsze warstwy: crossover po kolumnach
                layer_size = w1.shape[1]
                crossover_point = random.randint(1, layer_size - 1)
                new_w = np.hstack([
                    w1[:, :crossover_point],
                    w2[:, crossover_point:]
                ])
                new_b = np.concatenate([
                    b1[:, :crossover_point],
                    b2[:, crossover_point:]
                ], axis=1)

            new_weights.append(new_w.copy())
            new_biases.append(new_b.copy())

        child = self.generate_individual()
        child.set_weights(new_weights, new_biases)
        return child

    def mutate(self, individual, mutation_rate=0.3, mutation_strength=0.1):
        weights, biases = individual.get_weights()

        for i in range(len(weights)):
            for row in range(weights[i].shape[0]):
                for col in range(weights[i].shape[1]):
                    if random.random() < mutation_rate:
                        weights[i][row, col] += np.random.normal(0, mutation_strength)

        for i in range(len(biases)):
            for j in range(biases[i].shape[0]):
                if random.random() < mutation_rate:
                    biases[i][j] += np.random.normal(0, mutation_strength)

        individual.set_weights(weights, biases)
        return individual
    
    def select_parents(self, k=5):
        tournament = random.sample(self.generation, k)
        parents = sorted(tournament, key=self._evaluate_fitness)
        return parents[:2]  # Return the two best individuals from the tournament

        
    def run(self, generations=10, mutation_rate=0.1, k=5, verbose=True):
        best_fit = max(self.generation, key=self._evaluate_fitness)
        self.best_fitness = self._evaluate_fitness(best_fit)
        self.fitness_history.append(self.best_fitness)
        self.best_individual = best_fit
        if verbose:
            print(f"Generation 0: Best fitness = {self.best_fitness}")

        for generation in range(1, generations + 1):
            candidates = []
            while len(candidates) < self.population_size:
                parent1, parent2 = self.select_parents(k)
                child = self._crossover(parent1, parent2)
                child = self.mutate(child, mutation_rate)
                candidates.append(child)

            combined = self.generation + candidates
            combined.sort(key=self._evaluate_fitness)
            self.generation = combined[:self.population_size]  # Keep the best individuals
            best_fit = self.generation[0]
            best_fitness = self._evaluate_fitness(best_fit)
            self.fitness_history.append(best_fitness)
            if best_fitness < self.best_fitness:
                self.best_fitness = best_fitness
                self.best_individual = best_fit
            if verbose:
                print(f"Generation {generation}: Best fitness = {self.best_fitness}")
        return self.best_individual

    def plot_fitness(self):
        plt.plot(self.fitness_history)
        plt.xlabel('Generation')
        plt.ylabel('Best Fitness')
        plt.title('Fitness Evolution Over Generations')
        plt.grid()
        plt.show()

In [95]:
model = NeuralNetworkEvolution([1, 10, 10, 1], X_multimodal_train, y_multimodal_train,
                                 activation_fun="sigmoid", output_activation="linear", loss_fun="mse",
                                 population_size=100)
model.run(generations=10, mutation_rate=0.2, k=5, verbose=True)

Generation 0: Best fitness = 5400.887659828186
Generation 1: Best fitness = 5312.493124136593
Generation 2: Best fitness = 5312.493124136593
Generation 3: Best fitness = 5312.493124136593
Generation 4: Best fitness = 5312.493124136593
Generation 5: Best fitness = 5310.104846171927
Generation 6: Best fitness = 5310.104846171927
Generation 7: Best fitness = 5310.104846171927
Generation 8: Best fitness = 5310.104846171927
Generation 9: Best fitness = 5310.104846171927
Generation 10: Best fitness = 5303.4321377229835
